In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Path setup relative to NBA_ML_Project/notebooks/
DATA_DIR = "../data"
MODEL_DIR = "../models"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Tunable parameters (NBA equivalent of your NFL 3-week rolling window)
EWMA_SPAN = 10          # 10-game exponentially weighted window (~3 weeks of NBA play)
MIN_GAMES = 5           # Minimum games played before generating a prediction
EDGE_THRESHOLD = 1.5    # Minimum point disagreement vs. BetMGM to trigger a bet

In [ ]:
# Load your static team game logs and historical BetMGM spreads
team_logs_df = pd.read_csv(f"{DATA_DIR}/nba_team_game_logs.csv", parse_dates=["GAME_DATE"])
betmgm_odds_df = pd.read_csv(f"{DATA_DIR}/betmgm_historical_spreads.csv")

print(f"Loaded {len(team_logs_df):,} team-game rows and {len(betmgm_odds_df):,} BetMGM spread records.")
display(team_logs_df.head(3))

In [ ]:
df_sorted = team_logs_df.sort_values(["TEAM_ID", "GAME_DATE"]).copy()

core_metrics = ["NET_RATING", "EFG_PCT", "TM_TOV_PCT", "OREB_PCT", "FTA_RATE", "PACE"]

for col in core_metrics:
    # .shift(1) guarantees tonight's box score never leaks into tonight's pre-game features
    df_sorted[f"{col}_EWMA_{EWMA_SPAN}"] = (
        df_sorted.groupby("TEAM_ID")[col]
        .transform(lambda x: x.shift(1).ewm(span=EWMA_SPAN, min_periods=MIN_GAMES).mean())
    )

# Calculate rest days and back-to-back flags
df_sorted["REST_DAYS"] = (
    df_sorted.groupby("TEAM_ID")["GAME_DATE"]
    .diff()
    .dt.days.fillna(3)
    .clip(upper=4)
)
df_sorted["IS_B2B"] = (df_sorted["REST_DAYS"] == 1).astype(int)

# Sanity check: Verify for a single team that row N's EWMA only uses games up to N-1
sample_cols = ["GAME_DATE", "MATCHUP", "NET_RATING", f"NET_RATING_EWMA_{EWMA_SPAN}", "REST_DAYS", "IS_B2B"]
display(df_sorted[df_sorted["TEAM_ABBREVIATION"] == "IND"][sample_cols].head(8))

In [ ]:
home_df = df_sorted[df_sorted["MATCHUP"].str.contains("vs.")].copy()
away_df = df_sorted[df_sorted["MATCHUP"].str.contains("@")].copy()

matchups = pd.merge(home_df, away_df, on="GAME_ID", suffixes=("_HOME", "_AWAY"))
matchups = pd.merge(matchups, betmgm_odds_df[["GAME_ID", "BETMGM_HOME_SPREAD"]], on="GAME_ID", how="inner")

# Target Variable & BetMGM Benchmark
matchups["ACTUAL_HOME_MARGIN"] = matchups["PTS_HOME"] - matchups["PTS_AWAY"]
matchups["BETMGM_EXPECTED_MARGIN"] = -matchups["BETMGM_HOME_SPREAD"]

# Feature Differentials (Home minus Away)
feature_cols = []
for col in core_metrics:
    diff_col = f"DIFF_{col}_EWMA_{EWMA_SPAN}"
    matchups[diff_col] = matchups[f"{col}_EWMA_{EWMA_SPAN}_HOME"] - matchups[f"{col}_EWMA_{EWMA_SPAN}_AWAY"]
    feature_cols.append(diff_col)

matchups["DIFF_REST_DAYS"] = matchups["REST_DAYS_HOME"] - matchups["REST_DAYS_AWAY"]
matchups["B2B_DISADVANTAGE"] = matchups["IS_B2B_HOME"] - matchups["IS_B2B_AWAY"]
feature_cols.extend(["DIFF_REST_DAYS", "B2B_DISADVANTAGE"])

# Drop early-season rows where EWMA is NaN
matchups = matchups.dropna(subset=feature_cols + ["ACTUAL_HOME_MARGIN", "BETMGM_HOME_SPREAD"])
matchups = matchups.sort_values("GAME_DATE_HOME").reset_index(drop=True)

print(f"Clean Matchups Ready for Modeling: {len(matchups):,}")
display(matchups[["GAME_DATE_HOME", "MATCHUP_HOME", "ACTUAL_HOME_MARGIN", "BETMGM_HOME_SPREAD"] + feature_cols[:3]].tail(5))

In [ ]:
X = matchups[feature_cols].values
y = matchups["ACTUAL_HOME_MARGIN"].values

tscv = TimeSeriesSplit(n_splits=5)
model_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

oof_frames = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    model_pipeline.fit(X[train_idx], y[train_idx])
    fold_df = matchups.iloc[test_idx].copy()
    fold_df["FOLD"] = fold
    fold_df["PRED_HOME_MARGIN"] = model_pipeline.predict(X[test_idx])
    oof_frames.append(fold_df)

results = pd.concat(oof_frames, ignore_index=True)

# Inspect standardized Ridge coefficients from the final fold
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Ridge_Weight": model_pipeline.named_steps["ridge"].coef_
}).sort_values("Ridge_Weight", ascending=False)

display(coef_df)

In [ ]:
# Compare overall RMSE
model_rmse = np.sqrt(mean_squared_error(results["ACTUAL_HOME_MARGIN"], results["PRED_HOME_MARGIN"]))
betmgm_rmse = np.sqrt(mean_squared_error(results["ACTUAL_HOME_MARGIN"], results["BETMGM_EXPECTED_MARGIN"]))

# Calculate Model Edge & ATS Result
results["MODEL_HOME_EDGE"] = results["PRED_HOME_MARGIN"] + results["BETMGM_HOME_SPREAD"]
results["ATS_MARGIN"] = results["ACTUAL_HOME_MARGIN"] + results["BETMGM_HOME_SPREAD"]

print(f"Out-of-Sample Games: {len(results):,} | BetMGM RMSE: {betmgm_rmse:.3f} | Model RMSE: {model_rmse:.3f}\n")

# Sweep multiple edge thresholds to see where profitability peaks
summary_rows = []
for thresh in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    sub = results[results["MODEL_HOME_EDGE"].abs() >= thresh].copy()
    sub["BET_SIDE"] = np.where(sub["MODEL_HOME_EDGE"] >= thresh, "HOME", "AWAY")
    
    wins = (((sub["BET_SIDE"] == "HOME") & (sub["ATS_MARGIN"] > 0)) |
            ((sub["BET_SIDE"] == "AWAY") & (sub["ATS_MARGIN"] < 0))).sum()
    losses = (((sub["BET_SIDE"] == "HOME") & (sub["ATS_MARGIN"] < 0)) |
              ((sub["BET_SIDE"] == "AWAY") & (sub["ATS_MARGIN"] > 0))).sum()
    pushes = (sub["ATS_MARGIN"] == 0).sum()
    
    decided = wins + losses
    win_pct = (wins / decided * 100) if decided > 0 else 0.0
    units = (wins * (100 / 110)) - losses
    roi = (units / len(sub) * 100) if len(sub) > 0 else 0.0
    
    summary_rows.append({
        "Edge_Threshold (pts)": f">= {thresh}",
        "Bets": len(sub),
        "Record": f"{wins}W-{losses}L-{pushes}P",
        "Win_Pct": round(win_pct, 2),
        "Net_Units": round(units, 2),
        "ROI_Pct": round(roi, 2)
    })

display(pd.DataFrame(summary_rows))

# Save static outputs for Power BI dashboard & future inference
results.to_csv(f"{DATA_DIR}/nba_betmgm_backtest_results.csv", index=False)
joblib.dump(model_pipeline, f"{MODEL_DIR}/ridge_baseline_margin.joblib")